# 模組測試
每個 cell 獨立測試一個模組，按需執行。

In [1]:
# 環境設定（每次啟動先執行這格）
import sys, os
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
sys.path.insert(0, os.path.abspath('.'))
print('環境設定完成')

環境設定完成


In [2]:
# 測試資料重置（每次測試前先執行此格，清除 simulate.py 累積的狀態）
from utils.test_reset import reset_all
reset_all()
print('角色狀態已重置，可安全執行後續測試')

已重置 5 個角色：['A', 'B', 'C', 'D', 'E']
角色狀態已重置，可安全執行後續測試


## config/

In [3]:
from config.world_config import CHARACTER_NAMES, CHARACTER_CODES, AI_DATA_DIR
from config.model_config import MODEL_ID, INTUITIVE_MAX_TOKENS
from config.action_list import VALID_ACTIONS, VALID_LOCATIONS

print(f'CHARACTER_NAMES: {CHARACTER_NAMES}')
print(f'AI_DATA_DIR: {AI_DATA_DIR}')
print(f'MODEL_ID: {MODEL_ID}')
print(f'VALID_ACTIONS 共 {len(VALID_ACTIONS)} 個')
print(f'VALID_LOCATIONS 共 {len(VALID_LOCATIONS)} 個')

CHARACTER_NAMES: {'A': 'Amy', 'B': 'Ben', 'C': 'Claire', 'D': 'David', 'E': 'Emma'}
AI_DATA_DIR: c:\Users\Rayyu\Desktop\agi\NEW\project\AI_Data
MODEL_ID: microsoft/Phi-3.5-vision-instruct
VALID_ACTIONS 共 13 個
VALID_LOCATIONS 共 17 個


## utils/file_io.py

In [4]:
from utils.file_io import load_character, load_all_characters

data = load_character('A')
print(f'載入 A: name={data["name"]}, name_code={data["name_code"]}')

all_chars = load_all_characters()
print(f'載入全部角色: {list(all_chars.keys())}')

載入 A: name=Amy, name_code=A
載入全部角色: ['A', 'B', 'C', 'D', 'E']


## utils/logger.py

In [5]:
from utils.logger import get_logger, log_turn

logger = get_logger('test')
logger.info('logger 測試訊息')
log_turn(logger, 'A', 'D001_T001', '工作', 0.32, 'intuitive')
print('logger 測試完成，確認 logs/ 資料夾有產生 log 檔')

[17:45:44] [INFO] logger 測試訊息
[17:45:44] [INFO] [A] D001_T001 | mode=intuitive | C=0.320 | action=工作


logger 測試完成，確認 logs/ 資料夾有產生 log 檔


## core/character.py

In [6]:
from utils.file_io import load_character
from core.character import Character

data = load_character('A')
char = Character(data)

print(f'code={char.code}, name={char.name}, role={char.role}')
print(f'emotion={char.emotion}, day={char.day}, location={char.current_location}')
print(f'personality_short: {char.get_personality(short=True)}')
print(f'A 對 D 的關係（直覺）: {char.get_relationship_text("D", include_summary=False)}')

char.add_today_action('起床')
char.add_today_action('泡咖啡')
print(f'today_actions: {char.get_today_actions()}')

slot = char.get_current_slot()
print(f'當前時段: {slot}')
print(f'is_sleep_time: {char.is_sleep_time()}')

char.mark_slot_completed('07:00')
char.insert_dynamic_slot('14:00', '對話', '咖啡廳')
print(f'schedule: {[(s["time"], s["action"], s["type"]) for s in char.get_schedule()]}')

char.emotion = '開心'
char.current_location = '公園'
print(f'更新後 emotion={char.emotion}, location={char.current_location}')

_d_before = char.day
char.advance_day()
print(f'advance_day 後 day={char.day}（從{_d_before}+1）, today_actions={char.get_today_actions()}')
assert char.day == _d_before + 1

code=A, name=Amy, role=咖啡師
emotion=平靜, day=1, location=公寓
personality_short: 溫柔體貼但壓抑自己，習慣照顧他人，不輕易表露真實情緒。
A 對 D 的關係（直覺）: Amy暗自欣賞David，覺得他可靠穩重。但Amy擔心表達感情會讓David尷尬或影響他工作，所以從未說出口。
today_actions: ['起床', '泡咖啡']
當前時段: {'time': '07:00', 'action': '起床泡咖啡', 'location': '公寓', 'type': 'fixed', 'completed': False}
is_sleep_time: False
schedule: [('07:00', '起床泡咖啡', 'fixed'), ('08:30', '前往咖啡廳開店', 'fixed'), ('12:00', '午休', 'fixed'), ('14:00', '對話', 'dynamic'), ('18:00', '打烊整理', 'fixed'), ('19:00', '回公寓', 'fixed'), ('23:00', '睡覺', 'fixed')]
更新後 emotion=開心, location=公園
advance_day 後 day=2（從1+1）, today_actions=[]


## core/stm.py

In [7]:
from utils.file_io import load_character
from core.stm import STM

data = load_character('A')
stm = STM(data)

print(f'初始筆數: {stm.count()}, 容量: {stm.capacity()}')
print(f'is_full: {stm.is_full()}')

for i in range(1, 4):
    tid = STM.make_turn_id(day=1, turn_number=i)
    stm.add_turn(
        turn_id=tid,
        scene=f'咖啡廳，早上 {7+i} 點',
        image_desc=f'場景描述 {i}',
        input_text=f'輸入文字 {i}',
        action='工作',
        ham_propositions=[{'subject': 'Amy', 'relation': '工作', 'object': '咖啡廳'}]
    )

print(f'寫入 3 筆後 count={stm.count()}')
print(f'get_recent(2): {[t["turn_id"] for t in stm.get_recent(2)]}')
print(f'all_propositions: {len(stm.get_all_propositions())} 筆')
print(f'make_turn_id(1,5): {STM.make_turn_id(1, 5)}')
print(f'next_turn_number: {stm.next_turn_number()}')
stm.clear()
print(f'clear 後 count={stm.count()}')

初始筆數: 0, 容量: 15
is_full: False
寫入 3 筆後 count=3
get_recent(2): ['D001_T002', 'D001_T003']
all_propositions: 3 筆
make_turn_id(1,5): D001_T005
next_turn_number: 4
clear 後 count=0


## core/ltm.py

In [8]:
from utils.file_io import load_character
from core.ltm import LTM

data = load_character('A')
data['ltm']['propositions'] = []  # 確保乾淨初始狀態
data['ltm']['ltm_summary']  = ''
ltm = LTM(data)

print(f'初始命題數: {ltm.count()}')

ltm.encode('Amy', '遇見', 'David', location='咖啡廳', time='早上', day=1)
ltm.encode('Amy', '欣賞', 'David', day=1)
ltm.encode('David', '每天買', '咖啡', location='咖啡廳', day=1)
print(f'encode 3 筆後: {ltm.count()}')

ltm.encode_batch([
    {'subject': 'Amy', 'relation': '工作', 'object': '咖啡廳'},
    {'subject': 'Ben', 'relation': '追求', 'object': 'Amy'},
], day=1)
print(f'encode_batch 2 筆後: {ltm.count()}')

results = ltm.retrieve(query_subject='Amy')
print(f'retrieve(subject=Amy): {len(results)} 筆')
print(f'to_text:\n{ltm.to_text(results)}')

ltm.set_summary('Amy曾在咖啡廳遇見David，內心有好感但未表達。')
print(f'summary: {ltm.get_summary()}')

ltm.apply_decay()
print(f'decay 後 strengths: {[round(p["strength"],3) for p in ltm.get_all()]}')

removed = ltm.prune()
print(f'prune 後移除 {removed} 筆，剩餘 {ltm.count()} 筆')

初始命題數: 0
encode 3 筆後: 3
encode_batch 2 筆後: 5
retrieve(subject=Amy): 3 筆
to_text:
- Amy 遇見 David（咖啡廳，早上）
- Amy 欣賞 David
- Amy 工作 咖啡廳
summary: Amy曾在咖啡廳遇見David，內心有好感但未表達。
decay 後 strengths: [0.967, 0.967, 0.95, 0.967, 0.95]
prune 後移除 0 筆，剩餘 5 筆


## core/confusion.py

In [9]:
from core.confusion import evaluate

weights = {'w1': 0.4, 'w2': 0.3, 'w3': 0.3, 'threshold': 0.45}

# 情境1：平靜日常 -> 直覺
r1 = evaluate(
    image_desc='咖啡廳內，早上，無特殊狀況',
    input_text='David：早安。',
    current_action='工作',
    current_scene='咖啡廳，早上九點',
    ltm_summary='Amy曾與David短暫交談過。',
    today_actions=['起床', '開店'],
    weights=weights,
)
print(f'情境1（平靜）   C={r1["C"]} mode={r1["mode"]}')

# 情境2：驚訝 -> 介於中間
r2 = evaluate(
    image_desc='第一次見到的陌生人站在門口',
    input_text='陌生人：請問這裡是Amy的店嗎？',
    current_action='工作',
    current_scene='陌生人突然到訪',
    ltm_summary='Amy每天在咖啡廳工作，認識David、Emma等。',
    today_actions=['起床', '開店'],
    weights=weights,
)
print(f'情境2（驚訝）   C={r2["C"]} mode={r2["mode"]}')

# 情境3：K+S同時觸發 -> 思考
r3 = evaluate(
    image_desc='陌生人突然闖入',
    input_text='陌生人：緊急！快離開！',
    current_action='休息',
    current_scene='陌生人突然出現，情況緊急',
    ltm_summary='Amy每天在咖啡廳工作。',
    today_actions=[],
    weights=weights,
)
print(f'情境3（K+S觸發）C={r3["C"]} mode={r3["mode"]}')
print(f'  U={r3["U"]} K={r3["K"]} S={r3["S"]}')

情境1（平靜）   C=0.0 mode=intuitive
情境2（驚訝）   C=0.21 mode=intuitive
情境3（K+S觸發）C=0.57 mode=deliberate
  U=0.0 K=1.0 S=0.9


## core/memory_consolidation.py

In [ ]:
from utils.file_io import load_character
from core.character import Character
from core.stm import STM
from core.ltm import LTM
from core.memory_consolidation import consolidate

data_a = load_character('A')
char = Character(data_a)
stm  = STM(data_a)
ltm  = LTM(data_a)

for i in range(1, 4):
    stm.add_turn(
        turn_id=STM.make_turn_id(1, i),
        scene=f'咖啡廳，早上{7+i}點',
        image_desc='杯子',
        input_text='David點了一杯拿鐵。' if i == 2 else '',
        action='工作',
        ham_propositions=[{'subject': 'Amy', 'relation': '工作', 'object': '咖啡廳'}]
    )

print(f'STM 寫入 {stm.count()} 筆，LTM 初始 {ltm.count()} 筆')

call_log = []
def mock_model_fn(prompt: str) -> str:
    call_log.append(prompt[:30])
    if '值得長期' in prompt or '選出值得' in prompt:
        return '[{"subject":"Amy","relation":"工作","object":"咖啡廳"}]'
    if '總結' in prompt or '摘要' in prompt:
        return 'Amy每天在咖啡廳工作。'
    if '關係' in prompt:
        return '兩人關係友好。'
    return '平靜'

def mock_make_fn(**kw):
    return mock_model_fn

day_before = char.day
result = consolidate(char, stm, ltm, mock_make_fn)

print(f'新增命題: {result["new_propositions"]}')
print(f'LTM 總計: {result["ltm_total"]}')
print(f'修剪筆數: {result["ltm_pruned"]}')
print(f'摘要: {result["summary"]}')
print(f'STM 清空後 count={stm.count()}')
print(f'情緒: {char.emotion}, day={char.day}（從 {day_before} 推進）')
print(f'模型被呼叫 {len(call_log)} 次')

assert stm.count() == 0, 'STM 應被清空'
assert char.day == day_before + 1, f'day應推進 +1，{day_before}→{char.day}'
assert isinstance(result["new_propositions"], int)
print('memory_consolidation 測試全部通過')

## model/output_parser.py

In [11]:
from model.output_parser import parse_output

# 測試1：標準三區塊
raw1 = """[ACTION]: 前往:咖啡廳
[THOUGHT]: Amy想去看看David是否在。
[HAM]: [{"subject":"Amy","relation":"前往","object":"咖啡廳"}]
[/HAM]"""
r1 = parse_output(raw1)
assert r1['verb']   == '前往'
assert r1['target'] == '咖啡廳'
assert len(r1['ham']) == 1
print(f'測試1通過: action={r1["action"]}, ham={len(r1["ham"])}筆')

# 測試2：對話行動
raw2 = """[ACTION]: 對話:早安，今天天氣不錯！
[THOUGHT]: 想跟David打招呼。
[HAM]: []
[/HAM]"""
r2 = parse_output(raw2)
assert r2['verb']   == '對話'
assert r2['target'] == '早安，今天天氣不錯！'
print(f'測試2通過: verb={r2["verb"]}, target={r2["target"][:12]}')

# 測試3：inline HAM（無 [/HAM]）
raw3 = """[ACTION]: 工作
[THOUGHT]: 繼續工作。
[HAM]: [{"subject":"Amy","relation":"工作","object":"咖啡廳"},{"subject":"Amy","relation":"服務","object":"客人"}]"""
r3 = parse_output(raw3)
assert r3['verb'] == '工作'
assert len(r3['ham']) == 2
print(f'測試3通過: verb={r3["verb"]}, ham={len(r3["ham"])}筆')

# 測試4：fallback
r4 = parse_output('模型胡亂輸出了廢話')
assert r4['verb'] == '休息'
assert r4['ham'] == []
print(f'測試4通過（fallback）: verb={r4["verb"]}')

# 測試5：多命題
raw5 = """[ACTION]: 對話:你最近還好嗎？
[THOUGHT]: Claire想確認David的狀態。
[HAM]: [
  {"subject":"Claire","relation":"關心","object":"David"},
  {"subject":"David","relation":"壓力大","object":"工作"}
]
[/HAM]"""
r5 = parse_output(raw5)
assert len(r5['ham']) == 2
print(f'測試5通過: ham={r5["ham"]}')
print('output_parser 全部測試通過')

測試1通過: action=前往:咖啡廳, ham=1筆
測試2通過: verb=對話, target=早安，今天天氣不錯！
測試3通過: verb=工作, ham=2筆
測試4通過（fallback）: verb=休息
測試5通過: ham=[{'subject': 'Claire', 'relation': '關心', 'object': 'David'}, {'subject': 'David', 'relation': '壓力大', 'object': '工作'}]
output_parser 全部測試通過


## model/prompt_builder.py

In [12]:
from utils.file_io import load_character
from core.character import Character
from core.stm import STM
from core.ltm import LTM
from model.prompt_builder import PromptBuilder

data = load_character('A')
char = Character(data)
stm  = STM(data)
ltm  = LTM(data)

for i in range(1, 4):
    stm.add_turn(
        turn_id=STM.make_turn_id(1, i),
        scene=f'咖啡廳，第{i}輪',
        image_desc='杯子',
        input_text=f'David說了第{i}句話',
        action='工作',
        ham_propositions=[{'subject':'Amy','relation':'遇見','object':'David'}]
    )

ltm.encode('Amy', '欣賞', 'David', location='咖啡廳', day=1)
ltm.encode('Amy', '工作', '咖啡廳', day=1)
ltm.set_summary('Amy每天在咖啡廳工作，內心欣賞David。')

pb = PromptBuilder(char, stm, ltm)

# 直覺路徑
p_int = pb.build(scene='咖啡廳，早上九點', mode='intuitive', target_code='D')
assert '[ACTION]' in p_int
assert char.name in p_int
assert '咖啡廳，早上九點' in p_int
print(f'直覺 prompt 長度: {len(p_int)} 字元')

# 思考路徑
p_del = pb.build(scene='咖啡廳，早上九點', mode='deliberate', target_code='D')
assert '[ACTION]' in p_del
assert len(p_del) >= len(p_int)
print(f'思考 prompt 長度: {len(p_del)} 字元（> 直覺 {len(p_int)}）')

# 無對象
p_none = pb.build(scene='辦公室', mode='deliberate', target_code=None)
print(f'無對象 deliberate 長度: {len(p_none)} 字元')

# 驗證 access_count 不被重複計數（fix 驗證）
props = ltm.get_all()
max_ac = max((p['access_count'] for p in props), default=0)
print(f'LTM 最大 access_count = {max_ac}（應 <= 1）')
assert max_ac <= 1, f'access_count 不應超過 1，實際={max_ac}'
print('prompt_builder 測試通過')

直覺 prompt 長度: 651 字元
思考 prompt 長度: 833 字元（> 直覺 651）
無對象 deliberate 長度: 763 字元
LTM 最大 access_count = 1（應 <= 1）
prompt_builder 測試通過


## world/world_clock.py

In [13]:
from world.world_clock import WorldClock

clock = WorldClock(start_time='07:00', minutes_per_tick=30)
assert clock.time_str == '07:00' and clock.day == 1
print(f'初始: {clock.scene_prefix()}')

clock.tick(); assert clock.time_str == '07:30'
clock.tick(); assert clock.time_str == '08:00'
print(f'2次tick後: {clock.scene_prefix()}')

# 跨日
fast = WorldClock(start_time='23:30', minutes_per_tick=60)
fast.tick()
assert fast.day == 2 and fast.time_str == '00:30'
print(f'跨日: day={fast.day}, time={fast.time_str}')

# advance_day
clock.advance_day()
assert clock.day == 2 and clock.time_str == '07:00'
print(f'advance_day後: {clock.scene_prefix()}')

# should_trigger_slot
slot_past   = {'time': '06:30', 'action': '起床'}  # 06:30 < clock(07:00)
slot_future = {'time': '20:00', 'action': '睡覺'}
assert clock.should_trigger_slot(slot_past)   == True   # 07:00 >= 06:30
assert clock.should_trigger_slot(slot_future) == False
print(f'should_trigger_slot: 過去={True}, 未來={False}')

# is_sleep_time
from utils.file_io import load_character
from core.character import Character
data = load_character('A')
char = Character(data)
sleep_clock = WorldClock(start_time='23:00', minutes_per_tick=30)
print(f'23:00 is_sleep_time for A: {sleep_clock.is_sleep_time(char)}')

# get_pending_slots
pending = clock.get_pending_slots(char)
print(f'get_pending_slots（08:00時）: {len(pending)} 個時段')

print('world_clock 測試通過')

初始: 第1天 07:00
2次tick後: 第1天 08:00
跨日: day=2, time=00:30
advance_day後: 第2天 07:00
should_trigger_slot: 過去=True, 未來=False
23:00 is_sleep_time for A: False
get_pending_slots（08:00時）: 1 個時段
world_clock 測試通過


## perception/yolo_handler.py

In [14]:
from perception.yolo_handler import YoloHandler
from PIL import Image
import numpy as np

yolo = YoloHandler()
print(f'YOLO 模型載入: {yolo._model is not None}（False = fallback 模式）')

dummy = Image.fromarray(np.zeros((480, 640, 3), dtype=np.uint8))
dets = yolo.detect(dummy)
print(f'detect（fallback）: {dets}')

# is_meaningful
assert yolo.is_meaningful([]) == False
assert yolo.is_meaningful([{'class':'dog','confidence':0.9,'bbox':[0,0,10,10]}]) == False
assert yolo.is_meaningful([{'class':'person','confidence':0.9,'bbox':[0,0,10,10]}]) == True
print('is_meaningful 測試通過')

# to_description
fake = [
    {'class':'person','confidence':0.9,'bbox':[0,0,10,10]},
    {'class':'person','confidence':0.8,'bbox':[5,5,15,15]},
    {'class':'cup',   'confidence':0.7,'bbox':[0,0,5,5]},
    {'class':'car',   'confidence':0.9,'bbox':[0,0,20,20]},  # 不在清單
]
desc = yolo.to_description(fake, location='咖啡廳')
print(f'to_description: {desc}')
assert '咖啡廳' in desc and '人' in desc and '杯子' in desc and 'car' not in desc

# process（fallback）
ok, d2 = yolo.process(dummy, location='超市')
assert ok == False and d2 == ''
print(f'process（fallback）: meaningful={ok}, desc={repr(d2)}')
print('yolo_handler 測試通過')

YOLO 模型載入: True（False = fallback 模式）
detect（fallback）: []
is_meaningful 測試通過
to_description: 咖啡廳裡有2個人、杯子。
process（fallback）: meaningful=False, desc=''
yolo_handler 測試通過


## agent/agent.py（假模型）

In [15]:
from utils.file_io import load_character
from core.character import Character
from core.stm import STM
from core.ltm import LTM
from agent.agent import Agent
from config.action_list import VALID_ACTIONS

# 假模型
class MockFusion:
    def fuse_inputs(self, text, image_inputs, **kw): return {}
    def generate(self, fused_inputs, gen_cfg=None):
        return ('[ACTION]: 工作\n'
                '[THOUGHT]: Amy繼續在咖啡廳工作，心情平靜。\n'
                '[HAM]: [{"subject":"Amy","relation":"工作","object":"咖啡廳"}]\n'
                '[/HAM]')

class MockText:
    @staticmethod
    def build_prompt(t, num_images=0, system_text=None):
        return type('P', (), {'prompt': t})()

class MockVision:
    @staticmethod
    def encode(imgs): return type('V', (), {'to_dict': lambda s: {}})()

class MockLoader:
    fusion = MockFusion(); text = MockText(); vision = MockVision()
    def make_model_fn(self, **kw):
        def fn(p):
            if '值得' in p: return '[{"subject":"Amy","relation":"工作","object":"咖啡廳"}]'
            if '摘要' in p: return 'Amy每天工作。'
            if '關係' in p: return '友好。'
            return '平靜'
        return fn

data  = load_character('A')
char  = Character(data)
stm   = STM(data)
ltm   = LTM(data)
agent = Agent(char, stm, ltm, MockLoader())

result = agent.step(
    scene='咖啡廳，早上九點，陽光充足',
    image=None,
    input_text='David走進來點了拿鐵。',
    image_desc='咖啡廳裡有1個人。',
    target_code='D',
)

print(f'action:  {result["action"]}')
print(f'verb:    {result["verb"]}')
print(f'thought: {result["thought"]}')
print(f'mode:    {result["mode"]}  C={result["confusion"]["C"]:.3f}')
print(f'ham:     {result["ham"]}')
print(f'should_sleep: {result["should_sleep"]}')

assert result['verb'] in VALID_ACTIONS,   f'verb {result["verb"]} 不在清單'
assert result['mode'] in ('intuitive','deliberate')
assert 0 <= result['confusion']['C'] <= 1
assert isinstance(result['ham'], list)
assert isinstance(result['should_sleep'], bool)
assert stm.count() == 1,                  f'STM 應有1筆，得{stm.count()}'
assert char.current_action == result['verb']
print(f'STM count={stm.count()}, char.action={char.current_action}')

# 睡眠濃縮
_day_before = char.day
sr = agent.sleep()
assert stm.count() == 0, 'sleep後STM應清空'
assert char.day == _day_before + 1, f'day應推進+1，{_day_before}→{char.day}'
print(f'sleep: new_props={sr["new_propositions"]}, ltm_total={sr["ltm_total"]}, day={char.day}（+1）')
print('agent.py 測試全部通過')

[17:45:52] [INFO] [A] 開始睡眠濃縮...
[17:45:52] [INFO] [A] 濃縮完成：新增 1 筆 LTM，修剪 0 筆，總計 1 筆


action:  工作
verb:    工作
thought: Amy繼續在咖啡廳工作，心情平靜。
mode:    intuitive  C=0.000
ham:     [{'subject': 'Amy', 'relation': '工作', 'object': '咖啡廳'}]
should_sleep: False
STM count=1, char.action=工作
sleep: new_props=1, ltm_total=1, day=2（+1）
agent.py 測試全部通過


## agent/agent_manager.py（假模型）

In [16]:
from agent.agent_manager import AgentManager
from world.world_clock import WorldClock
from config.action_list import VALID_ACTIONS

# 複用 MockLoader（需先執行上方 agent.py 那格，或在此重定義）
class MockFusion:
    def fuse_inputs(self, text, image_inputs, **kw): return {}
    def generate(self, fi, gen_cfg=None):
        return ('[ACTION]: 工作\n[THOUGHT]: 繼續工作。\n'
                '[HAM]: [{"subject":"Amy","relation":"工作","object":"咖啡廳"}]\n[/HAM]')
class MockText:
    @staticmethod
    def build_prompt(t, num_images=0, system_text=None): return type('P',(),{'prompt':t})()
class MockVision:
    @staticmethod
    def encode(imgs): return type('V',(),{'to_dict': lambda s:{}})()
class MockLoader:
    fusion=MockFusion(); text=MockText(); vision=MockVision()
    def make_model_fn(self,**kw):
        def fn(p):
            if '值得' in p: return '[{"subject":"Amy","relation":"工作","object":"咖啡廳"}]'
            if '摘要' in p: return 'Amy工作。'
            if '關係' in p: return '友好。'
            return '平靜'
        return fn

clock   = WorldClock(start_time='07:00', minutes_per_tick=30)
manager = AgentManager(loader=MockLoader(), clock=clock)
clock.tick()

print(f'角色: {manager.all_codes()}')
print(f'時間: {clock.scene_prefix()}')

# step_character
r = manager.step_character(code='A', scene='咖啡廳', input_text='早安。')
assert r['verb'] in VALID_ACTIONS
print(f'A step: action={r["action"]}, mode={r["mode"]}')

# step_all
results = manager.step_all(scene='上午的世界')
assert len(results) == 5
print(f'step_all: { {k: v["action"] for k,v in results.items()} }')

# 防遞迴驗證：_forward_dialogue 傳給 B，B 的回應不再反向傳給 A
call_log = []
orig_b = manager._agents['B'].step
def tracked_b(**kw):
    call_log.append('B')
    return orig_b(**kw)
manager._agents['B'].step = tracked_b

manager._forward_dialogue('A', 'B', '早安！', '咖啡廳')
print(f'B 被呼叫 {call_log.count("B")} 次（防遞迴：應 <= 1）')
assert call_log.count('B') <= 1, '防遞迴失效'

# WorldClock / Character day 同步驗證
_a_day_pre = manager.get_character('A').day
_clk_pre   = clock.day
print(f'睡眠前: char.day={_a_day_pre}, clock.day={_clk_pre}')
manager._do_sleep('A')
_a_day_post = manager.get_character('A').day
print(f'A 睡眠後: char.day={_a_day_post}（+1）')
assert _a_day_post == _a_day_pre + 1, f'day應+1，{_a_day_pre}→{_a_day_post}'

print('agent_manager.py 測試全部通過')

[17:45:52] [INFO] AgentManager 初始化，角色：['A', 'B', 'C', 'D', 'E']
[17:45:52] [INFO] [A] D001_T001 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [A] D001_T002 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [B] D001_T001 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [C] D001_T001 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [D] D001_T001 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [E] D001_T001 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [A→B] 對話傳遞
[17:45:52] [INFO] [B] D001_T002 | mode=intuitive | C=0.000 | action=工作
[17:45:52] [INFO] [A] 開始睡眠濃縮...
[17:45:52] [INFO] [A] 濃縮完成：新增 1 筆 LTM，修剪 0 筆，總計 1 筆
[17:45:52] [INFO] [A] Day 2 睡眠濃縮完成 | STM=1 筆 -> LTM 新增後共 1 筆命題
[17:45:52] [INFO] [A] 存檔完成，進入第 2 天


角色: ['A', 'B', 'C', 'D', 'E']
時間: 第1天 07:30
A step: action=工作, mode=intuitive
step_all: {'A': '工作', 'B': '工作', 'C': '工作', 'D': '工作', 'E': '工作'}
B 被呼叫 1 次（防遞迴：應 <= 1）
睡眠前: char.day=1, clock.day=1
A 睡眠後: char.day=2（+1）
agent_manager.py 測試全部通過
